# Mitsui GBM: One Model Per Target

Features per model (~40 total):
- **Autoregressive (train_labels):** past values of the target at safe offsets `{horizon+1, horizon+2, horizon+5, horizon+10, horizon+20}`. The minimum shift is `horizon+1` so the AR feature window ends before the prediction window starts — no overlap, no leakage.
- **Fundamentals (train.csv):** lagged 1-day returns of 9 key commodity/FX prices + copper/gold ratio at lags {1, 5, 20}.

424 models, 3-seed averaged.

In [21]:
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import lightgbm as lgb

DATA_DIR      = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data'
HOLDOUT_START = 1827
LAGS_AR       = [1, 2, 5, 10, 20]   # autoregressive lags of the target itself
LAGS_FUND     = [1, 5, 20]          # lags for fundamental return features
VAL_FRAC      = 0.15
SEEDS         = (0, 1, 2)

# Key commodity/FX columns from train.csv
FUNDAMENTAL_COLS = [
    'LME_AH_Close',                         # aluminum
    'LME_ZS_Close',                         # zinc
    'LME_CA_Close',                         # copper
    'LME_PB_Close',                         # lead
    'JPX_Gold_Standard_Futures_Close',      # gold
    'JPX_Platinum_Standard_Futures_Close',  # platinum
    'FX_EURUSD',
    'FX_USDJPY',
    'FX_AUDUSD',                            # commodity-currency proxy
]

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')

print(f'train {train.shape}  labels {train_labels.shape}')

train (1961, 558)  labels (1961, 425)


In [22]:
# ── Parse utilities ───────────────────────────────────────────────────────────

def parse_pair(pair_str):
    s = pair_str.strip()
    tokens = re.split(r'\s*([+-])\s*', s)
    result, sign = [], 1
    if tokens[0]:
        result.append((sign, tokens[0]))
    i = 1
    while i < len(tokens):
        op, ticker = tokens[i], tokens[i + 1]
        result.append((1 if op == '+' else -1, ticker))
        i += 2
    return result

def get_exchange(ticker):
    if ticker.startswith('US_Stock_'):
        return 'US_Stock'
    return ticker.split('_')[0]

# Build target_info
rows = []
for _, r in target_pairs.iterrows():
    components = parse_pair(r['pair'])
    tickers    = [t for _, t in components]
    exchanges  = sorted(set(get_exchange(t) for t in tickers))
    rows.append({
        'target':       r['target'],
        'lag':          r['lag'],
        'pair_raw':     r['pair'],
        'tickers':      tickers,
        'exchange_key': '_'.join(exchanges),
    })
target_info = pd.DataFrame(rows)
print(target_info.head())

     target  lag                                        pair_raw  \
0  target_0    1                           US_Stock_VT_adj_close   
1  target_1    1            LME_PB_Close - US_Stock_VT_adj_close   
2  target_2    1                     LME_CA_Close - LME_ZS_Close   
3  target_3    1                     LME_AH_Close - LME_ZS_Close   
4  target_4    1  LME_AH_Close - JPX_Gold_Standard_Futures_Close   

                                           tickers  exchange_key  
0                          [US_Stock_VT_adj_close]      US_Stock  
1            [LME_PB_Close, US_Stock_VT_adj_close]  LME_US_Stock  
2                     [LME_CA_Close, LME_ZS_Close]           LME  
3                     [LME_AH_Close, LME_ZS_Close]           LME  
4  [LME_AH_Close, JPX_Gold_Standard_Futures_Close]       JPX_LME  


In [23]:
# ── Align and sort both dataframes on date_id ─────────────────────────────────

train        = train.sort_values('date_id').reset_index(drop=True)
train_labels = train_labels.sort_values('date_id').reset_index(drop=True)

# Sanity: date_ids should match after inner-joining
assert (train['date_id'].values == train_labels['date_id'].values).all(), \
    'date_id mismatch between train and train_labels'

target_cols = [c for c in train_labels.columns if c != 'date_id']
n_days      = len(train)
print(f'{n_days} days, {len(target_cols)} targets')

1961 days, 424 targets


In [24]:
# ── Precompute fundamental feature block (shared across all target models) ────
#
# AR features are NOT precomputed here because the safe shift depends on each
# target's prediction horizon. They are built per-target in the training loop.

fund_parts = []
for col in FUNDAMENTAL_COLS:
    ret = train[col].pct_change(1)          # 1-day return for stationarity
    for k in LAGS_FUND:
        fund_parts.append(ret.shift(k).rename(f'fund_{col}_ret_lag{k}'))

copper_gold = train['LME_CA_Close'] / (train['JPX_Gold_Standard_Futures_Close'] + 1e-8)
for k in LAGS_FUND:
    fund_parts.append(copper_gold.shift(k).rename(f'fund_copper_gold_ratio_lag{k}'))

fund_df = pd.concat(fund_parts, axis=1).ffill().fillna(0)

# Lookup: target name → its prediction horizon (lag field from target_pairs)
lag_for_target = target_info.set_index('target')['lag'].to_dict()

n_fund_features = fund_df.shape[1]
n_ar_features   = len(LAGS_AR)
print(f'fund features: {n_fund_features}, AR features per model: {n_ar_features}')
print(f'Total features per model: {n_fund_features + n_ar_features}')

fund features: 30, AR features per model: 5
Total features per model: 35


In [25]:
# ── Scoring metric (ddof=0 matches official Kaggle scorer) ────────────────────

def mitsui_metric(y_true, y_pred, verbose=False):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    arr = np.array(daily_corrs)
    std = arr.std(ddof=0)
    if std == 0:
        raise ZeroDivisionError('Daily corr std is zero.')
    sharpe = arr.mean() / std
    if verbose:
        print(f'  mean={arr.mean():.4f}  std={std:.4f}  sharpe={sharpe:.4f}  n_days={len(arr)}')
    return sharpe

In [26]:
# ── Train/holdout split indices ───────────────────────────────────────────────

train_mask   = train['date_id'] < HOLDOUT_START
holdout_mask = train['date_id'] >= HOLDOUT_START

n_train   = train_mask.sum()
split_idx = int(n_train * (1 - VAL_FRAC))   # within-train val boundary

# Holdout ground truth (preserving NaN for scoring mask)
y_hold_true = (
    train_labels.loc[holdout_mask, target_cols].values
)

print(f'n_train={n_train}, val split at row {split_idx}, holdout={holdout_mask.sum()} days')

n_train=1827, val split at row 1552, holdout=134 days


In [27]:
# ── Train one LightGBM per target ─────────────────────────────────────────────
#
# For target with prediction horizon k, AR shifts = [k+1, k+2, k+5, k+10, k+20].
# This guarantees the AR feature window ends at t-1, before the target window
# starts at t — eliminating the overlap leakage from the naive shift(1) approach.

fund_arr = fund_df.values.astype(np.float32)
fund_train   = fund_arr[train_mask.values]
fund_holdout = fund_arr[holdout_mask.values]

labels_arr = train_labels[target_cols].values   # (n_days, 424), may contain NaN

predictions = np.zeros((holdout_mask.sum(), len(target_cols)), dtype=np.float32)

lgbm_base_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    verbose=-1,
)

for t_idx, tname in enumerate(target_cols):
    horizon     = lag_for_target[tname]          # 1, 2, 3, or 4
    safe_shifts = [horizon + j for j in LAGS_AR] # e.g. lag-1 target → [2,3,6,11,21]

    col_series = train_labels[tname]
    ar_block   = np.column_stack([
        col_series.shift(s).fillna(0).values for s in safe_shifts
    ]).astype(np.float32)   # (n_days, n_ar_features)

    ar_train   = ar_block[train_mask.values]
    ar_holdout = ar_block[holdout_mask.values]

    X_full = np.hstack([ar_train, fund_train])
    y_full = pd.Series(labels_arr[:, t_idx]).loc[train_mask].fillna(0).values.astype(np.float32)

    X_tr, X_val = X_full[:split_idx], X_full[split_idx:]
    y_tr, y_val = y_full[:split_idx], y_full[split_idx:]

    X_hold = np.hstack([ar_holdout, fund_holdout])

    seed_preds = np.zeros(len(X_hold), dtype=np.float32)
    for seed in SEEDS:
        m = lgb.LGBMRegressor(**lgbm_base_params, random_state=seed)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(40, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        seed_preds += m.predict(X_hold)
    predictions[:, t_idx] = seed_preds / len(SEEDS)

    if (t_idx + 1) % 50 == 0:
        print(f'trained {t_idx + 1}/{len(target_cols)}')

print('Done.')

/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

trained 50/424


/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

trained 100/424


/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

...

/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

trained 400/424


/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

Done.


/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [28]:
# ── Score ─────────────────────────────────────────────────────────────────────

print('=== Holdout Score ===')
score = mitsui_metric(y_hold_true, predictions, verbose=True)
print(f'\nFinal holdout Sharpe: {score:.4f}')

=== Holdout Score ===
  mean=0.0406  std=0.1767  sharpe=0.2297  n_days=134

Final holdout Sharpe: 0.2297


In [29]:
# ── Per-exchange-group breakdown ──────────────────────────────────────────────

print('=== Per-group Sharpe ===')
for ekey in sorted(target_info['exchange_key'].unique()):
    group_targets = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    y_idx   = [target_cols.index(t) for t in group_targets]
    g_true  = y_hold_true[:, y_idx]
    g_pred  = predictions[:, y_idx]
    try:
        s = mitsui_metric(g_true, g_pred)
        print(f'  {ekey:<20} {s:+.4f}  ({len(group_targets)} targets)')
    except ZeroDivisionError as e:
        print(f'  {ekey:<20} ERROR: {e}')

=== Per-group Sharpe ===
  FX                   -0.0299  (2 targets)
  FX_JPX               +0.2863  (46 targets)
  FX_LME               +0.0841  (86 targets)
  JPX_LME              +0.1049  (12 targets)
  JPX_US_Stock         +0.0133  (87 targets)
  LME                  +0.1155  (8 targets)
  LME_US_Stock         +0.1310  (181 targets)
  US_Stock             +0.1022  (2 targets)
